# Train and Predict Drones

### Abstract

...


### Introduction

...


### Dataset



# 🏁 1. Initialization

pip install `ultralytics` and [dependencies](https://github.com/ultralytics/ultralytics/blob/main/pyproject.toml) and check software and hardware.

[![PyPI - Version](https://img.shields.io/pypi/v/ultralytics?logo=pypi&logoColor=white)](https://pypi.org/project/ultralytics/) [![PyPI - Python Version](https://img.shields.io/pypi/pyversions/ultralytics?logo=python&logoColor=gold)](https://pypi.org/project/ultralytics/)

### 1.0 Installing dependencies

In [1]:
!pip install ultralytics markdown rich wrapt pandas huggingface_hub scikit-learn opencv-python wandb python-dotenv datasets -q

### 1.1 Importing Libraries

In [2]:
from datasets import load_dataset, Image, concatenate_datasets, DatasetDict
from IPython.display import display, Image as IPyImage
from sklearn.model_selection import KFold
from ultralytics import YOLO, settings
from typing import Iterable, Union
import matplotlib.pyplot as plt
from collections import Counter
from dotenv import load_dotenv
from datetime import datetime
from pathlib import Path
from PIL import Image

from tqdm import tqdm
import pandas as pd
import ultralytics
import numpy as np
import datetime
import random
import shutil
import wandb
import uuid
import yaml
import math
import re
import os

ultralytics.checks()

Ultralytics 8.4.8 🚀 Python-3.13.5 torch-2.9.1+cu128 CUDA:0 (NVIDIA A10, 22588MiB)
Setup complete ✅ (96 CPUs, 377.7 GB RAM, 4866.1/5037.1 GB disk)


### 1.2 Global Definitions

In [14]:
DATASET_ROOT_DIR = Path('./datasets/main')
DATASET_ALL_DIR = DATASET_ROOT_DIR / 'train_validation_test'
TRAINING_DATASET_DIRECTORY = DATASET_ROOT_DIR / 'train'
VALIDATION_DATASET_DIRECTORY = DATASET_ROOT_DIR / 'valid/'
TEST_DATASET_DIRECTORY = DATASET_ROOT_DIR / 'test/'

IMAGE_APPLY_GRAYSCALE = True

MODELS_DIRECTORY = Path('./models/')
load_dotenv()

False

### 1.3 Global Settings

In [4]:
# YOLO settings
settings.update({"wandb": True})

# Initialize Weights & Biases environment
wandb.login(key=os.getenv("WANDB_TOKEN"))

# login("TOKEN") # Keep commented if token loaded from .env file

wandb: ERROR Failed to detect the name of this notebook. You can set it manually with the WANDB_NOTEBOOK_NAME environment variable to enable code saving.
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /home/jovyan/.netrc.
wandb: Currently logged in as: cqsv20ajs (hibou) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

### 1.4 Global Structure

In [5]:
dataset_structure = {
    "root": Path(""),
    "name": "",
    "classes": [],
    "train": {
        "images": [],
        "labels": [],
    },
    "valid": {
        "images": [],
        "labels": [],
    },
    "test": {
        "images": [],
        "labels": [],
    },
}


### 1.5 Global Function Definitions

In [6]:
from itertools import chain


def list_files(
        directory: Union[str, Path],
        extensions: Iterable[str],
        include_root_directory: bool = False,
        recursive: bool = False,
) -> list[Path]:
    directory = Path(directory)
    extensions = tuple(extensions)

    matched_files = []

    if recursive:
        iterator = directory.rglob("*")
    else:
        iterator = directory.iterdir()

    for p in iterator:
        if p.is_file() and p.suffix in extensions:
            matched_files.append(
                p if include_root_directory else p.name
            )

    return matched_files


def numeric_key(name):
    """Extract the first number from a filename for sorting."""
    nums = re.findall(r'\d+', name)
    return int(nums[0]) if nums else float('inf')


def sort_files_by_number(files_to_sort: list):
    """
    Sort a list of filenames by the first number found in each name.

    Args:
        files_to_sort (list): List of filenames (strings)

    Returns:
        list: Sorted list of filenames
    """
    return sorted(files_to_sort, key=lambda i: int(i.stem))


def update_dataset_structure():
    dataset_structure["root"] = Path(DATASET_ROOT_DIR)
    dataset_structure["name"] = DATASET_ROOT_DIR.name

    dataset_structure["train"]["images"] = sort_files_by_number(
        list_files(TRAINING_DATASET_DIRECTORY, [".jpg", ".jpeg", ".JPG", ".JPEG"], True))
    dataset_structure["train"]["labels"] = sort_files_by_number(list_files(TRAINING_DATASET_DIRECTORY, [".txt"], True))

    dataset_structure["valid"]["images"] = sort_files_by_number(
        list_files(VALIDATION_DATASET_DIRECTORY, [".jpg", ".jpeg", ".JPG", ".JPEG"], True))
    dataset_structure["valid"]["labels"] = sort_files_by_number(
        list_files(VALIDATION_DATASET_DIRECTORY, [".txt"], True))

    dataset_structure["test"]["images"] = sort_files_by_number(
        list_files(VALIDATION_DATASET_DIRECTORY, [".jpg", ".jpeg", ".JPG", ".JPEG"], True))
    dataset_structure["test"]["labels"] = sort_files_by_number(list_files(VALIDATION_DATASET_DIRECTORY, [".txt"], True))

    dataset_structure["classes"] = ["drone", "other"]


def delete_files_in_dataset(files_to_delete: list):
    try:
        confirm = input("Files are going to be deleted. Type 'yes' to continue: ").strip().lower()
        if confirm != 'yes':
            print("Deletion aborted by user.")
            return

        for file in files_to_delete:
            if os.path.isfile(file):
                os.remove(file)
                print(f"Deleted: {file}")
            else:
                print(f"Warning: File does not exist: {file}")

    except KeyboardInterrupt:
        print("\nDeletion aborted by user (KeyboardInterrupt).")
    finally:
        try:
            update_dataset_structure()
        except NameError:
            pass


def backup_dataset():
    dataset_path = dataset_structure.get("path", "")
    backup_dir = os.path.join(dataset_path, "backup")
    os.makedirs(backup_dir, exist_ok=True)

    now = datetime.now().strftime('%Y-%m-%d_%H-%M-%S')
    target_name = f"{dataset_structure["name"]}-{now}"
    target_path = os.path.join(backup_dir, target_name)

    def ignore_backup(_, names):
        return {"backup"} if "backup" in names else set()

    shutil.copytree(dataset_path, target_path, ignore=ignore_backup)
    print(f"Backup created at: {target_path}")
    return str(target_path)


def interpret_map(map_value: float) -> str:
    """
    Interpret mAP value according to standard object detection heuristics.
    """
    if map_value < 0.10:
        return "Model is effectively failing"
    elif map_value < 0.30:
        return "Very weak performance"
    elif map_value < 0.50:
        return "Usable baseline"
    elif map_value < 0.70:
        return "Good performance"
    else:
        return "Strong performance"

def plot_image_grid(images_path, nb_cols = 4, max_images_preview = -1, show_title = False):
    if max_images_preview != -1:
        images_path = images_path[:max_images_preview]

    rows = math.ceil(len(images_path) / nb_cols)

    img = Image.open(images_path[0])
    w, h = img.size  # pixels

    dpi = 100

    img_w = w / dpi
    img_h = h / dpi
    #
    plt.figure(figsize=(nb_cols * img_w, rows * img_h))

    for i, path in enumerate(images_path):
        img = Image.open(path)
        plt.subplot(rows, nb_cols, i + 1)
        plt.imshow(img)
        plt.axis("off")
        if show_title:
            plt.title(path.name, fontsize=25)

    plt.tight_layout()
    plt.show()

# 📂 2. Dataset

### 2.0 Acquire Dataset

Download dataset from hugging face.

In [8]:
dataset = load_dataset("Hibou-Foundation/computer-vision")

### 2.1 Split dataset

Split the dataset into train, validation, test.

In [9]:
base_split = "train_validation_test"
label_column = "class_id"

train_ratio = [0.8, 0.8]  # [class 0, class 1]
valid_ratio = [0.1, 0.1]
test_ratio = [0.1, 0.1]

seed = 42

for i in range(len(train_ratio)):
    assert train_ratio[i] + valid_ratio[i] + test_ratio[i] == 1.0

train_parts = []
valid_parts = []
test_parts = []

num_classes = len(train_ratio)

for cls in range(num_classes):
    cls_ds = dataset[base_split].filter(
        lambda x: x[label_column] == cls
    )

    cls_ds = cls_ds.shuffle(seed=seed)

    n = len(cls_ds)
    n_train = int(n * train_ratio[cls])
    n_valid = int(n * valid_ratio[cls])

    train_parts.append(cls_ds.select(range(0, n_train)))
    valid_parts.append(cls_ds.select(range(n_train, n_train + n_valid)))
    test_parts.append(cls_ds.select(range(n_train + n_valid, n)))

train_ds = concatenate_datasets(train_parts).shuffle(seed=seed)
valid_ds = concatenate_datasets(valid_parts).shuffle(seed=seed)
test_ds = concatenate_datasets(test_parts).shuffle(seed=seed)

dataset = DatasetDict({
    "train": train_ds,
    "validation": valid_ds,
    "test": test_ds,
})
dataset

DatasetDict({
    train: Dataset({
        features: ['image', 'class_id', 'class_name', 'box', 'name', 'raw_label'],
        num_rows: 661
    })
    validation: Dataset({
        features: ['image', 'class_id', 'class_name', 'box', 'name', 'raw_label'],
        num_rows: 3489
    })
    test: Dataset({
        features: ['image', 'class_id', 'class_name', 'box', 'name', 'raw_label'],
        num_rows: 464
    })
})

### 2.2 Convert to file images

In [10]:
# Create folders
for split in ["train", "valid", "test"]:
    os.makedirs(f"{DATASET_ROOT_DIR}/{split}", exist_ok=True)


def export_to_yolo(ds, split_name):
    for idx, sample in enumerate(ds):
        image = sample["image"]  # already a PIL.Image
        label = sample["raw_label"]  # already YOLO format [[class, cx, cy, w, h], ...]
        img_name = sample["name"]
        txt_name = img_name.split(".")[0] + ".txt"

        # Save image
        img_path = f"{DATASET_ROOT_DIR}/{split_name}/{img_name}"
        image.save(img_path, quality=95)

        # Save labels
        lbl_path = f"{DATASET_ROOT_DIR}/{split_name}/{txt_name}"
        with open(lbl_path, "w") as f:
            f.write(label)


# Run export
split_mapping = {"train": "train", "validation": "valid", "test": "test"}
for hf_split, folder_name in split_mapping.items():
    export_to_yolo(dataset[hf_split], folder_name)
update_dataset_structure()

# 📚️ 3. Models Settings

### 3.1 Model selection

In [11]:
selected_size = "nano"
selected_version = "26"

YOLO_MODEL_SIZE = {
    "nano": "n",
    "small": "s",
    "medium": "m",
    "large": "l",
    "xlarge": "x",
}
run_session_id = str(uuid.uuid4()).split("-")[0]  # For wandb
model_name = f"yolo{selected_version}{YOLO_MODEL_SIZE[selected_size]}.pt"
model_path = MODELS_DIRECTORY / model_name
model = YOLO(model_path, task="detect")
print(f"Session ID: {run_session_id}")

Session ID: 6d9549b8


### 3.2 Training Configuration
Define hyperparameters (epochs, batch size, image size)

In [15]:
train_config = {
    'epochs': 170,
    'imgsz': 640,
    'batch': 16,
    'lr0': 0.01,
    'patience': 20,
    'optimizer': 'auto',
    'project': 'computer-vision',
    'device': [0]
}
train_config

{'epochs': 170,
 'imgsz': 640,
 'batch': 16,
 'lr0': 0.01,
 'patience': 20,
 'optimizer': 'auto',
 'project': 'computer-vision',
 'device': [0]}

**Parameter breakdown:**

- **train**: Executes the YOLOv11x training pipeline.
- **model**=yolov11x.pt: Uses pre-trained YOLOv11x weights as initialization.
- **data**=/content/data.yaml: Specifies the dataset configuration file.
- **imgsz**=640: Sets input resolution to enhance small-object detection.
- **lr0**=0.001: Sets the learning rate for each training step.
- **epochs**=32: Defines the number of training cycles over the dataset.
- **batch**=16: Sets the batch size for each training step.
- **device**=0: Allocates GPU device 0 for training.
- **optimizer**=AdamW: As default, AdamW optimization algorithm utilized.

# 🔍️ 4. Hyperparameter Tuning

- [Official documentation](https://docs.ultralytics.com/guides/hyperparameter-tuning/)

## 4.1 Genetic Algorithm

In [15]:
# Define search space
search_space = {
    "lr0": (1e-5, 1e-1),
    "degrees": (0.0, 45.0),
}
resume_tuning = False

In [16]:
data_yaml = dict(
    train=os.path.join('../../', TRAINING_DATASET_DIRECTORY),
    val=os.path.join('../../', VALIDATION_DATASET_DIRECTORY),
    nc=2,
    channels = 1 if IMAGE_APPLY_GRAYSCALE else 3,
    names=['drone', 'other']
)

data_config_path = DATASET_ROOT_DIR / 'data.yaml'

with open(data_config_path, 'w') as outfile:
    yaml.dump(data_yaml, outfile, default_flow_style=True)

%cat "$data_config_path"

{channels: 3, names: [drone, other], nc: 2, train: ../../datasets/main/train, val: ../../datasets/main/valid}


In [ ]:
model.tune(
    data=data_config_path,
    epochs=30,
    iterations=300,
    optimizer="AdamW",
    space=search_space,
    plots=False,
    save=False,
    val=False,
    resume=resume_tuning,
)

## 4.2 K-Fold Cross Validation

#### 4.2.1 Load labels and classes

In [ ]:
update_dataset_structure()

train_path = TRAINING_DATASET_DIRECTORY
valid_path = VALIDATION_DATASET_DIRECTORY
test_path = TEST_DATASET_DIRECTORY

labels = list(chain(dataset_structure["train"]["labels"], dataset_structure["valid"]["labels"]))

cls_idx = list(range(len(dataset_structure["classes"])))
classes = list(dataset_structure["classes"])

#### 4.2.2 Init empty pandas DataFram

In [ ]:
index = [label.stem for label in labels]  # uses base filename as ID (no extension)
labels_df = pd.DataFrame([], columns=cls_idx, index=index)

#### 4.2.3 Count the instances of each class-label present in the annotation files.

In [ ]:
for label in labels:
    lbl_counter = Counter()

    with open(label) as lf:
        lines = lf.readlines()

    for line in lines:
        # classes for YOLO label use integer at the first position of each line
        lbl_counter[int(line.split(" ", 1)[0])] += 1

    labels_df.loc[label.stem] = lbl_counter

labels_df = labels_df.fillna(0.0).infer_objects(copy=False)  # replace `nan` values with `0.0`
labels_df

#### 4.2.4 Configure K-Fold Dataset Split

In [ ]:
random.seed(0)  # for reproducibility
k_split = 5
kf = KFold(n_splits=k_split, shuffle=True, random_state=20)  # setting random_state for repeatable results

k_folds = list(kf.split(labels_df))

#### 4.2.5 Do something

In [ ]:
folds = [f"split_{n}" for n in range(1, k_split + 1)]
folds_df = pd.DataFrame(index=index, columns=folds)

for i, (train, val) in enumerate(k_folds, start=1):
    col = f"split_{i}"

    folds_df.loc[labels_df.iloc[train].index, col] = "train"
    folds_df.loc[labels_df.iloc[val].index, col] = "val"


#### 4.2.6 Do something

In [ ]:
fold_lbl_distrb = pd.DataFrame(index=folds, columns=cls_idx)

for n, (train_indices, val_indices) in enumerate(k_folds, start=1):
    train_totals = labels_df.iloc[train_indices].sum()
    val_totals = labels_df.iloc[val_indices].sum()

    # To avoid division by zero, we add a small value (1E-7) to the denominator
    ratio = val_totals / (train_totals + 1e-7)
    fold_lbl_distrb.loc[f"split_{n}"] = ratio

#### 4.2.7 Create the directories

In [ ]:
# Initialize an empty list to store image file paths
images = list(chain(dataset_structure["train"]["images"], dataset_structure["valid"]["images"]))

# Create the necessary directories and dataset YAML files
save_path = Path(DATASET_ROOT_DIR / f"{datetime.date.today().isoformat()}_{k_split}-Fold_Cross-val")
save_path.mkdir(parents=True, exist_ok=True)
ds_yamls = []

for split in folds_df.columns:
    # Create directories
    split_dir = save_path / split
    split_dir.mkdir(parents=True, exist_ok=True)
    (split_dir / "train").mkdir(parents=True, exist_ok=True)
    (split_dir / "val").mkdir(parents=True, exist_ok=True)

    # Create dataset YAML files
    dataset_yaml = split_dir / f"{split}_dataset.yaml"
    ds_yamls.append(dataset_yaml)

    with open(dataset_yaml, "w") as ds_y:
        yaml.safe_dump(
            {
                "path": split_dir.as_posix(),
                "train": "train",
                "val": "val",
                "names": classes,
            },
            ds_y,
        )

#### 4.2.8 Copy the files

In [ ]:
for image, label in tqdm(zip(images, labels), total=len(images), desc="Copying files"):
    for split, k_split in folds_df.loc[image.stem].items():
        # Destination directory
        img_to_path = save_path / str(split) / k_split
        lbl_to_path = save_path / str(split) / k_split

        # Copy image and label files to a new directory (SamefileError if a file already exists)
        shutil.copy(image, img_to_path / image.name)
        shutil.copy(label, lbl_to_path / label.name)

In [ ]:
results = {}

for k, dataset_yaml in enumerate(ds_yamls):
    fold_run_name = f"{selected_version}-{selected_size}-{run_session_id}-fold_{k + 1}"

    model = YOLO(model_path, task="detect")
    results[k] = model.train(data=dataset_yaml, name=fold_run_name, **train_config)

In [ ]:
cv_metrics = {}

for k, dataset_yaml in enumerate(ds_yamls):
    model_path = Path(results[k].save_dir) / "weights/best.pt"
    fold_run_name = f"{selected_version}-{selected_size}-{run_session_id}-val-fold_{k + 1}"

    print(fold_run_name)

    validation_model = YOLO(model_path)
    metrics = validation_model.val(data=dataset_yaml, name=fold_run_name, **train_config, plots=True)

    cv_metrics[k] = {
        "mAP50": metrics.box.map50,
        "mAP75": metrics.box.map75,
        "mAP50-95": metrics.box.map,
        "per_class": metrics.box.maps,
    }

In [ ]:
map50 = np.mean([m["mAP50"] for m in cv_metrics.values()])
map75 = np.mean([m["mAP75"] for m in cv_metrics.values()])
map5095 = np.mean([m["mAP50-95"] for m in cv_metrics.values()])

map50_std = np.std([m["mAP50"] for m in cv_metrics.values()])
map75_std = np.std([m["mAP75"] for m in cv_metrics.values()])
map5095_std = np.std([m["mAP50-95"] for m in cv_metrics.values()])

print(f"mAP50:\t\t{map50:.4f} ± {map50_std:.4f} → " f"{interpret_map(float(map50))}")
print(f"mAP75:\t\t{map75:.4f} ± {map75_std:.4f} → "f"{interpret_map(float(map75))}")
print(f"mAP50-95:\t{map5095:.4f} ± {map5095_std:.4f} → " f"{interpret_map(float(map5095))}")

# ⚙️ 5. Training

Purpose: Handle model setup and training configuration.

### 5.1 Save dataset configuration

In [16]:
data_yaml = dict(
    train=os.path.join('../../', TRAINING_DATASET_DIRECTORY),
    val=os.path.join('../../', VALIDATION_DATASET_DIRECTORY),
    nc=2,
    channels = 1 if IMAGE_APPLY_GRAYSCALE else 3,
    names=['drone', 'other']
)

data_config_path = DATASET_ROOT_DIR / 'data.yaml'

with open(data_config_path, 'w') as outfile:
    yaml.dump(data_yaml, outfile, default_flow_style=True)
%cat "$data_config_path"

{channels: 1, names: [drone, other], nc: 2, train: ../../datasets/main/train, val: ../../datasets/main/valid}


### 5.2 Run Training

In [ ]:
run_name = f"{selected_version}-{selected_size}-{run_session_id}"
results = model.train(**train_config,
                      data=data_config_path,
                      name=run_name)

Ultralytics 8.4.8 🚀 Python-3.13.5 torch-2.9.1+cu128 CUDA:0 (NVIDIA A10, 22588MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=datasets/main/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=170, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=models/yolo26n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=26-nano-6d9549b8, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=20, per

Overriding model.yaml nc=80 with nc=2

                   from  n    params  module                                       arguments                     
  0                  -1  1       176  ultralytics.nn.modules.conv.Conv             [1, 16, 3, 2]                 
  1                  -1  1      4672  ultralytics.nn.modules.conv.Conv             [16, 32, 3, 2]                
  2                  -1  1      6640  ultralytics.nn.modules.block.C3k2            [32, 64, 1, False, 0.25]      
  3                  -1  1     36992  ultralytics.nn.modules.conv.Conv             [64, 64, 3, 2]                
  4                  -1  1     26080  ultralytics.nn.modules.block.C3k2            [64, 128, 1, False, 0.25]     
  5                  -1  1    147712  ultralytics.nn.modules.conv.Conv             [128, 128, 3, 2]              
  6                  -1  1     87040  ultralytics.nn.modules.block.C3k2            [128, 128, 1, True]           
  7                  -1  1    295424  ultralytics

[W129 13:01:10.579311815 NNPACK.cpp:56] Could not initialize NNPACK! Reason: Unsupported hardware.
[W129 13:01:10.186206405 NNPACK.cpp:56] Could not initialize NNPACK! Reason: Unsupported hardware.
[W129 13:01:10.279132035 NNPACK.cpp:56] Could not initialize NNPACK! Reason: Unsupported hardware.
[W129 13:01:11.383092009 NNPACK.cpp:56] Could not initialize NNPACK! Reason: Unsupported hardware.
[W129 13:01:11.494150694 NNPACK.cpp:56] Could not initialize NNPACK! Reason: Unsupported hardware.
[W129 13:01:11.667863738 NNPACK.cpp:56] Could not initialize NNPACK! Reason: Unsupported hardware.
[W129 13:01:11.776151335 NNPACK.cpp:56] Could not initialize NNPACK! Reason: Unsupported hardware.
[W129 13:01:11.871233112 NNPACK.cpp:56] Could not initialize NNPACK! Reason: Unsupported hardware.
[W129 13:01:11.184123955 NNPACK.cpp:56] Could not initialize NNPACK! Reason: Unsupported hardware.
[W129 13:01:11.275174620 NNPACK.cpp:56] Could not initialize NNPACK! Reason: Unsupported hardware.
[W129 13:0

YOLO26n summary: 260 layers, 2,504,292 parameters, 2,504,292 gradients, 5.7 GFLOPs



[W129 13:01:32.694422598 NNPACK.cpp:56] Could not initialize NNPACK! Reason: Unsupported hardware.
[W129 13:01:32.788324532 NNPACK.cpp:56] Could not initialize NNPACK! Reason: Unsupported hardware.
[W129 13:01:32.870312979 NNPACK.cpp:56] Could not initialize NNPACK! Reason: Unsupported hardware.
[W129 13:01:32.870842061 NNPACK.cpp:56] Could not initialize NNPACK! Reason: Unsupported hardware.


Transferred 606/708 items from pretrained weights
AMP: running Automatic Mixed Precision (AMP) checks...
AMP: checks passed ✅
train: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1966.3±1471.1 MB/s, size: 93.9 KB)
train: Scanning /home/jovyan/datasets/main/train... 108667 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 108667/108667 1.3Kit/s 1:27<0.0s
train: New cache created: /home/jovyan/datasets/main/train.cache
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1138.0±828.9 MB/s, size: 176.5 KB)
val: Scanning /home/jovyan/datasets/main/valid... 3489 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 3489/3489 1.3Kit/s 2.8s0.0s
val: New cache created: /home/jovyan/datasets/main/valid.cache
Plotting labels to /home/jovyan/runs/detect/computer-vision/26-nano-6d9549b8/labels.jpg... 
optimizer: 'optimizer=auto' found, ignoring 'lr0=0.01' and 'momentum=0.937' and determining best 'optimizer', 'lr0' and 'momentum' automatically... 
optimizer: MuSGD(lr=0.01, momentum=0.9) with parameter

### 5.3 Training results

In [ ]:
result_dir = Path(results.save_dir)
display(IPyImage(filename=str(result_dir / "results.png")))

In [ ]:
%matplotlib inline

# Retrieve val_batch images
val_batch = []
i = 0
batch_file = result_dir / f"val_batch{i}_labels.jpg"
while batch_file.exists():
    val_batch.append(batch_file)
    val_batch.append(result_dir / f"val_batch{i}_pred.jpg")
    i += 1
    batch_file = result_dir / f"val_batch{i}_labels.jpg"

# Retrieve confusion matrix images
confusion_matrix_path = [
    result_dir / "confusion_matrix.png",
    result_dir / "confusion_matrix_normalized.png"
]

# Retrieve metric images
boxes_path = [
    result_dir / "BoxF1_curve.png",
    result_dir / "BoxP_curve.png",
    result_dir / "BoxPR_curve.png",
    result_dir / "BoxR_curve.png",
]

# Show images
plot_image_grid(val_batch, nb_cols=2, show_title=True)
plot_image_grid(confusion_matrix_path, nb_cols=2)
plot_image_grid(boxes_path, nb_cols=2)


# ✅️ 6. Validation

###  6.1 Evaluate Model

In [ ]:
metrics = model.val()

In [ ]:
print(metrics.box.map)
print(metrics.box.map50)
print(metrics.box.map75)
print(metrics.box.maps)

# 🏭️ 7. Inference

Purpose: Evaluate results qualitatively and quantitatively.


### 7.0 Load model

In [ ]:
best_path = result_dir / "weights/best.pt"
# best_path = os.path.join("./computer-vision/nano11", "weights/best.pt")
custom_model = YOLO(best_path)

### 7.1 Run Predictions (IMAGES)

In [ ]:
predictions = custom_model.predict(
    TEST_DATASET_DIRECTORY,
    save=True,
    project=Path('computer-vision', run_name),
    conf=0.25
)
predictions_output_dir = predictions[0].save_dir

### 7.2 Visualize Predictions

In [ ]:
predictions_paths = list_files(predictions_output_dir, [".jpg", ".png", ".jpeg"], True)

plot_image_grid(predictions_paths, max_images_preview=100)

### 7.3 Run Predictions (VIDEOS)

In [ ]:
result = custom_model.track(
    source="https://www.youtube.com/watch?v=aZjRQfBWLEo",
    conf=0.3,
    iou=0.5,
    show=False,
    imgsz=640,
    save=True,
    project=Path('computer-vision', run_name),
    exist_ok=True
)

# ⛴️ 8. Export & Deployment

### Supported Formats

- [Source](https://docs.ultralytics.com/modes/export/#arguments)

| Format                                             | `format` Argument |
|----------------------------------------------------|-------------------|
| [PyTorch](https://pytorch.org/)                    | -                 |
| [TorchScript](../integrations/torchscript.md)      | `torchscript`     |
| [ONNX](../integrations/onnx.md)                    | `onnx`            |
| [OpenVINO](../integrations/openvino.md)            | `openvino`        |
| [TensorRT](../integrations/tensorrt.md)            | `engine`          |
| [CoreML](../integrations/coreml.md)                | `coreml`          |
| [TF SavedModel](../integrations/tf-savedmodel.md)  | `saved_model`     |
| [TF GraphDef](../integrations/tf-graphdef.md)      | `pb`              |
| [TF Lite](../integrations/tflite.md)               | `tflite`          |
| [TF Edge TPU](../integrations/edge-tpu.md)         | `edgetpu`         |
| [TF.js](../integrations/tfjs.md)                   | `tfjs`            |
| [PaddlePaddle](../integrations/paddlepaddle.md)    | `paddle`          |
| [MNN](../integrations/mnn.md)                      | `mnn`             |
| [NCNN](../integrations/ncnn.md)                    | `ncnn`            |
| [IMX500](../integrations/sony-imx500.md){{ tip3 }} | `imx`             |
| [RKNN](../integrations/rockchip-rknn.md)           | `rknn`            |
| [ExecuTorch](../integrations/executorch.md)        | `executorch`      |
| [Axelera](../integrations/axelera.md)              | `axelera`         |

In [ ]:
model.export(format="onnx")